# DATT Phase 3: YOLO11s PyTorch CUDA Validation on Google Colab

This notebook validates the Phase 3 migration of the **DATT - AI People Counter** detector from ONNX Runtime DirectML to **YOLO11s PyTorch (.pt) + Ultralytics + CUDA GPU**.

### Runtime Requirements:
- Google Colab Runtime: **GPU (T4, V100, or A100)**
- Menu: `Runtime` -> `Change runtime type` -> Hardware accelerator: `T4 GPU`

In [1]:
import torch

# Step 2: Check GPU and CUDA availability
cuda_available = torch.cuda.is_available()
print(f"CUDA available = {cuda_available}")

if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 2)
    print(f"Device Name: {device_name}")
    print(f"Device Count: {device_count}")
    print(f"VRAM Total: {total_memory:.2f} MB | VRAM Allocated: {allocated_memory:.2f} MB")
else:
    print("WARNING: No CUDA device found. Please switch to a GPU runtime.")

CUDA available = True
Device Name: Tesla T4
Device Count: 1
VRAM Total: 15102.00 MB | VRAM Allocated: 0.00 MB


In [2]:
# Step 3: Install minimal Colab dependencies
!pip install -q -r requirements-colab.txt

In [3]:
# Setup Python path
import os
import sys

repo_root = os.path.abspath(".")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import config
from src.detector.yolo_detector import YOLODetector
from src.tracker.bytetrack_tracker import PersonTracker
from src.counter.zone_counter import ZoneCounter
from src.utils.logger import logger

In [4]:
# Step 4: Initialize PyTorch CUDA YOLODetector
detector = YOLODetector(config)
print(f"Detector initialized successfully on device: {detector.device}")
print(f"Model: {detector.model_path.name}")

Detector initialized successfully on device: cuda:0
Model: yolo11s.pt


In [5]:
# Step 5 & 7: Test detection, ByteTrack, and ZoneCounter with PyTorch CUDA
import cv2
import numpy as np

test_img_path = "results/onnx_detection.jpg"
if os.path.isfile(test_img_path):
    frame = cv2.imread(test_img_path)
    frame = cv2.resize(frame, (config.WIDTH, config.HEIGHT))
else:
    frame = np.zeros((config.HEIGHT, config.WIDTH, 3), dtype=np.uint8)

tracker = PersonTracker(config)
counter = ZoneCounter(polygon=config.ZONE_POLYGON, person_class_id=config.PERSON_CLASS_ID)

# Warmup & inference
detections = detector.detect(frame)
tracks = tracker.update(detections)
people_in_view = counter.update(tracks, frame.shape)

print("Single Frame Test:")
print(f"Detections: {len(detections)}")
print(f"YOLO latency: {detector.last_yolo_ms:.1f} ms")
print(f"Tracks: {len(tracks)}")
print(f"People in view: {people_in_view}")

Single Frame Test:
Detections: 1
YOLO latency: 12.4 ms
Tracks: 1
People in view: 1


In [6]:
# Step 8: Benchmark 50 consecutive frames on CUDA
import time

latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    dets = detector.detect(frame)
    latencies.append((time.perf_counter() - t0) * 1000)

avg_lat = sum(latencies) / len(latencies)
print(f"Average CUDA YOLO Latency over 50 frames: {avg_lat:.2f} ms (~{1000 / avg_lat:.1f} FPS)")
if torch.cuda.is_available():
    print(f"CUDA VRAM Allocated: {torch.cuda.memory_allocated(0) / (1024 ** 2):.2f} MB")
    print(f"CUDA VRAM Reserved: {torch.cuda.memory_reserved(0) / (1024 ** 2):.2f} MB")